In [1]:

from azure.ai.ml.entities import AzureBlobDatastore
from azure.ai.ml.entities import AccountKeyConfiguration
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential

ml_client = MLClient.from_config(credential=DefaultAzureCredential())

print(f"Ready to use Azure ML SDK v2 to work with {ml_client.workspace_name}")

Found the config file in: /config.json


Ready to use Azure ML SDK v2 to work with ai-300


In [2]:
default_ds = ml_client.datastores.get_default()

for ds in ml_client.datastores.list():
    print(ds.name, "- Default =", ds.name == default_ds.name)

aml_datastore - Default = True
workspaceblobstore - Default = False
workspaceartifactstore - Default = False
workspaceworkingdirectory - Default = False
workspacefilestore - Default = False


In [3]:
aml_datastore = AzureBlobDatastore(
    name="aml_datastore",
    description="Datastore pointing to a blob container using https protocol.",
    account_name="amldatastoree",
    container_name="datastore",
    protocol="https",
    credentials=AccountKeyConfiguration(
        account_key="YOUR-ACCOUNT-KEY"
    ),
)

ml_client.create_or_update(aml_datastore)

HttpResponseError: (UserError) The datastore aml_datastore is currently the workspace default, so it cannot be updated with isDefault set to false. To change the default, update a different datastore with isDefault set to true.
Code: UserError
Message: The datastore aml_datastore is currently the workspace default, so it cannot be updated with isDefault set to false. To change the default, update a different datastore with isDefault set to true.
Additional Information:Type: ComponentName
Info: {
    "value": "managementfrontend"
}Type: Correlation
Info: {
    "value": {
        "operation": "9657c78403b96718d9f6db52012d4f5e",
        "request": "dcba0971cc62aed7"
    }
}Type: Environment
Info: {
    "value": "australiasoutheast"
}Type: Location
Info: {
    "value": "australiasoutheast"
}Type: Time
Info: {
    "value": "2026-04-22T10:47:28.1544887+00:00"
}Type: InnerError
Info: {
    "value": {
        "code": "BadArgument",
        "innerError": {
            "code": "ArgumentInvalid",

In [ ]:
import requests
from azure.ai.ml.entities import Data
from azure.ai.ml.constants import AssetTypes

# Upload file to datastore path
from azure.ai.ml.operations import DatastoreOperations

datastore = ml_client.datastores.get_default()

# Construct the datastore path
datastore_path = f"azureml://datastores/{datastore.name}/paths/diabetes.csv"

# Register the uploaded file as a data asset
data_asset = Data(
    path=datastore_path,
    type=AssetTypes.URI_FILE,
    name="diabetes-data",
    description="Diabetes dataset uploaded to datastore"
)

ml_client.data.create_or_update(data_asset)
     

In [ ]:
import mltable
from mltable import MLTableHeaders, MLTableFileEncoding
from azure.ai.ml.entities import Data
from azure.ai.ml.constants import AssetTypes

# Get datastore
datastore = ml_client.datastores.get_default()

#constructing File URI
file_uri = (
    f"azureml://subscriptions/{ml_client.subscription_id}"
    f"/resourcegroups/{ml_client.resource_group_name}"
    f"/workspaces/{ml_client.workspace_name}"
    f"/datastores/{datastore.name}"
    f"/paths/diabetes.csv"
)

# Path to CSV in datastore
paths = [{
    "file": file_uri
}]

# Create MLTable from CSV
tbl = mltable.from_delimited_files(
    paths=paths,
    delimiter=",",
    header=MLTableHeaders.all_files_same_headers,
    infer_column_types=True,
    encoding=MLTableFileEncoding.utf8
)

# Preview dataset
print(tbl.show())

# Save MLTable definition locally
mltable_folder = "./diabetes_mltable"
tbl.save(mltable_folder)

# Register MLTable as data asset
data_asset = Data(
    path=mltable_folder,
    type=AssetTypes.MLTABLE,
    name="diabetes-mltable",
    description="Diabetes dataset stored as MLTable"
)

ml_client.data.create_or_update(data_asset)